# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivor dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and inspect main dataset information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n\n{metadata['description']}")

## 2. Data Overview

List all available record sets and their fields. All references use `@id` as required for robust and reproducible access.

In [ ]:
# List all record sets defined in the metadata
from pprint import pprint

# RecordSet objects may be found at metadata['recordSet'] or metadata.recordSet in model
def list_record_sets(dataset):
    recs = dataset.metadata.recordSets
    if not recs:
        # Try the plural alternative
        recs = getattr(dataset.metadata, 'recordSet', None)
    if not recs:
        print("No record sets found.")
        return []
    if isinstance(recs, dict):
        recs = [recs]
    return recs

record_sets = list_record_sets(dataset)
if record_sets:
    print("Found record sets:")
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs.@id
        rs_name = rs.get('name', '(no name available)') if isinstance(rs, dict) else getattr(rs,'name', '(no name)')
        # List fields/columns for each record set
        print(f"- RecordSet @id: {rs_id} | Name: {rs_name}")
        # Try to find fields/columns for the record set
        fields = None
        if isinstance(rs, dict) and 'field' in rs:
            fields = rs['field']
        elif hasattr(rs, 'fields'):
            fields = rs.fields
        elif hasattr(rs, 'field'):
            fields = rs.field
        if fields:
            if isinstance(fields, dict):
                fields = [fields]
            print("  Fields:")
            for fld in fields:
                fld_id = fld['@id'] if isinstance(fld, dict) else fld.@id
                fld_name = fld.get('name', '(no name)') if isinstance(fld, dict) else getattr(fld, 'name', '(no name)')
                print(f"   - @id: {fld_id} | Name: {fld_name}")
        print()
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction

Load data from the available record sets using their `@id` fields. Data is loaded into Pandas DataFrames for ease of analysis.

In [ ]:
# Example: If record sets found, extract their @ids
# (Replace below with actual @ids discovered above; else use knowledge/example)
if record_sets:
    record_set_ids = [rs['@id'] if isinstance(rs, dict) else rs.@id for rs in record_sets]
else:
    # This dataset uses only one main record set, inferred from Croissant schemas:
    record_set_ids = ['https://sen.science/doi/10.71728/senscience.qs2f-h81p/recordSet/ClinicalData']

dataframes = dict()
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet: {record_set_id}")
            print("Columns:", df.columns.tolist())
            display(df.head(3))
        else:
            print(f"No records found in RecordSet: {record_set_id}")
    except Exception as e:
        print(f"Could not load RecordSet {record_set_id}:", e)

# For reference, print just the first available record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"Columns for {main_record_set_id}:\n", main_df.columns.tolist())
    display(main_df.head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Let's perform EDA by filtering, normalizing, and grouping data using field `@id`s. You may adapt the field `@id`s below depending on your inspection of the record set columns.

In [ ]:
# Example: Choose numeric and categorical field @ids from columns
import numpy as np
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # For this dataset, suppose 'AgeAtDiagnosis' and 'MSI_status' field @ids are present.
    # Update these as required based on real field @ids.
    available_fields = list(df.columns)
    print("Available columns:", available_fields)
    # Choose based on inspection:
    numeric_field_id = next((c for c in available_fields if 'Age' in c), available_fields[0])  # e.g., '@id:age_at_diagnosis'
    group_field_id = next((c for c in available_fields if 'MSI' in c and 'status' in c), None)
    print(f"Using field for numeric analysis: {numeric_field_id}")
    if group_field_id:
        print(f"Using group/categorical field: {group_field_id}")
    # Filter records: age greater than threshold
    threshold = 60
    filtered = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered.head())
    # Normalize
    norm_field = f"{numeric_field_id}_normalized"
    filtered[norm_field] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
    print(f"First five normalized ages:")
    display(filtered[[numeric_field_id, norm_field]].head())
    # Group by categorical (MSI status, or similar)
    if group_field_id and group_field_id in filtered.columns:
        grouped = filtered.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std'])
        print(f"Group statistics for {numeric_field_id} by {group_field_id}:")
        display(grouped)
else:
    print("No main DataFrame to analyze.")

## 5. Visualization

Let's visualize the age distribution and the MSI status breakdown. Adapt fields as necessary based on your available columns.

In [ ]:
# Visualization: histogram and bar chart
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=10, kde=True, color='royalblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.countplot(data=df, x=group_field_id, palette='Set2')
        plt.title(f'Count of Records by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel('Count')
        plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded and inspected the metadata of a FAIR² colorectal cancer survivor dataset via Croissant schema.
- Explored the available record sets and columns using their `@id` identifiers.
- Extracted tabular data and performed example filtering, normalization, and grouping.
- Visualized numeric field distributions and categorical breakdowns.

All field and record set references were made using `@id`. You can adapt this template for any Croissant-conformant dataset, simply by changing the schema URL and updating field/record set `@id` references as shown.